In [ ]:
import os
from pathlib import Path
import time

import weaviate
from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents.base import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_weaviate.vectorstores import WeaviateVectorStore
from weaviate.classes.init import Auth
from weaviate.classes.query import MetadataQuery


from rubin.rag.custom_weaviate_vector_store import CustomWeaviateVectorStore

# Connect to weaviate server

In [ ]:
# Add weaviate extentions to the proxy bypass list
os.environ["NO_PROXY"] = os.environ.get("NO_PROXY", "") + ",.svc,.cluster.local"
os.environ["no_proxy"] = os.environ["NO_PROXY"]

In [ ]:
def start_connect():
    client = weaviate.connect_to_custom(
    http_host=http_host,
    http_port=8080,  # Default is 80, WCD uses 443
    http_secure=False,
    grpc_host=grpc_host,
    grpc_port=50051,  # Default is 50051, WCD uses 443
    grpc_secure=False,
    auth_credentials=Auth.api_key(
        weaviate_api_key
    ),  # The API key to use for authentication
    headers={"X-OpenAI-Api-Key": openai_api_key},
    skip_init_checks=True,
    )
    return client

In [ ]:
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
weaviate_api_key = os.getenv("WEAVIATE_API_KEY")
http_host = "weaviate-headless.rubin-rag.svc.cluster.local"
grpc_host = "weaviate-grpc.rubin-rag.svc.cluster.local"

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set")
if weaviate_api_key is None:
    raise ValueError("WEAVIATE_API_KEY environment variable is not set")
if http_host is None:
    raise ValueError("HTTP_HOST environment variable is not set")
if grpc_host is None:
    raise ValueError("GRPC_HOST environment variable is not set")

client = start_connect()
try:
    print("Client is live:", client.is_live())
finally:
    client.close()

# Inspect weaviate instance

### list collections

In [ ]:
client = start_connect()
try:
    collections = client.collections.list_all()
    collection_names = list(collections.keys())
    print(collection_names)
    
finally:
    client.close()

### check instance meta

In [ ]:
client = start_connect()
try:
    meta = client.get_meta()
    print(meta)
    print("Weaviate version:", meta.get("version"))
finally:
    client.close()

# Inspect collection Ingestion_20250610

In [ ]:
client = start_connect()

try:
    # 1. Access the collection
    collection = client.collections.use("Ingestion_20250610")

    # 2. Retrieve its configuration
    config = collection.config.get(simple=False)

    # Inspect the Vectorizer
    print("Vectorizer:", config.vectorizer)
    print("Vectorizer Config:", config.vectorizer_config)

    print("Named vectors:", config.vector_config)

    # Inspect the Data Schema (Properties)
    print("\n--- Properties Schema ---")
    for p in config.properties:
        print(p.name, p.data_type)

    
finally:
    client.close()

### query from the collection

In [ ]:
client.close()
client = start_connect()
store = CustomWeaviateVectorStore(
      client=client,
      index_name="Ingestion_20250610",
      text_key="page_content",
      embedding=OpenAIEmbeddings(model="text-embedding-3-small",
  dimensions=1536),
)
collection = client.collections.use("Ingestion_20250610")
response2 = store.similarity_search("what is butler?", q=6)
client.close()

In [ ]:
response